# 🤖 Исследование AutoML-фреймворка AutoGluon и сравнение с FEDOT

**Тема работы:** Исследование AutoML-фреймворка AutoGluon и сравнение его с FEDOT на реальных задачах машинного обучения.

---

## Содержание

1. [Введение: что такое AutoML](#введение)
2. [Что такое AutoGluon и зачем он нужен](#autogluon)
3. [Методики под капотом](#методики)
4. [Эксперимент 1: Классификация (Adult Income)](#классификация)
5. [Эксперимент 2: Регрессия (California Housing)](#регрессия)
6. [Эксперимент 3: Временные ряды (AirPassengers)](#временные-ряды)
7. [Сравнение AutoGluon vs FEDOT](#сравнение)
8. [Выводы](#выводы)

---

> **Примечание:** Все три задачи используют публичные датасеты. FEDOT подключается через официальный pip-пакет. Время выполнения каждого эксперимента ограничено для Colab (~10–15 мин на блок).

## 0. Установка зависимостей

Устанавливаем AutoGluon, FEDOT и вспомогательные библиотеки. Выполните эту ячейку первой — займёт ~3–5 минут.

In [ ]:
# ── Установка AutoGluon (tabular + timeseries)
!pip install -q autogluon.tabular autogluon.timeseries

# ── Установка FEDOT
!pip install -q fedot

# ── Вспомогательные пакеты
!pip install -q scikit-learn pandas numpy matplotlib seaborn

In [ ]:
# ── Общие импорты
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)
import time

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('✅ Все базовые пакеты импортированы')

---

## 1. Введение: что такое AutoML <a name="введение"></a>

**AutoML (Automated Machine Learning)** — это класс методов и инструментов, который автоматизирует рутинные этапы разработки ML-моделей.

### Что автоматизирует AutoML:

| Этап | Описание |
|------|----------|
| Предобработка данных | Заполнение пропусков, кодирование, нормализация |
| Конструирование признаков | Генерация и отбор значимых признаков |
| Выбор алгоритма | Перебор и сравнение разных моделей |
| Подбор гиперпараметров | HPO — hyperparameter optimization |
| Ансамблирование | Объединение нескольких моделей |
| Оценка качества | Кросс-валидация, leaderboard |

### Ключевое разграничение:
- **AutoML** — это *подход*, а не конкретная программа
- **AutoGluon** — конкретный AutoML-фреймворк (Amazon)
- **FEDOT** — другой конкретный AutoML-фреймворк (NSS Lab, Россия)

---

## 2. Что такое AutoGluon и зачем он нужен <a name="autogluon"></a>

**AutoGluon** — open-source AutoML-библиотека от Amazon, разработанная в 2020 году.

Принцип работы:
1. Пользователь передаёт датасет, имя целевой колонки и лимит времени
2. AutoGluon автоматически обучает множество моделей
3. Строит несколько уровней стекинга
4. Возвращает финальный weighted ensemble

### Типы поддерживаемых задач:

| Задача | Поддержка | Примеры датасетов |
|--------|-----------|-------------------|
| Классификация (бинарная и мультикласс) | ✅ | Titanic, Adult Income |
| Регрессия | ✅ | California Housing, Bike Sharing |
| Временные ряды | ✅ | AirPassengers, M4, Electricity |
| Работа с текстом | ✅ | IMDb, AG News |
| Работа с изображениями | ✅ | CIFAR-10, Shopee |
| Multimodal (таблица + текст + изображение) | ✅ | Уникальная фича |

### Когда использовать AutoGluon:
- Нужен быстрый сильный baseline
- Нет времени на ручной подбор моделей
- Хочется сравнить много алгоритмов сразу
- Нужна воспроизводимость

---

## 3. Методики под капотом AutoGluon <a name="методики"></a>

### 3.1 Кросс-валидация

AutoGluon использует **bagging с out-of-fold (OOF) предсказаниями** — аналог k-fold CV:
- Каждая базовая модель обучается на (k-1) фолдах
- Предсказывает оставшийся фолд (out-of-fold prediction)
- OOF-предсказания используются как признаки для следующего уровня стекинга

### 3.2 Модели под капотом

In [ ]:
# ── Визуализация моделей и пайплайна AutoGluon

models_info = {
    'LightGBM':        {'type': 'Gradient Boosting', 'speed': 9, 'accuracy': 9},
    'CatBoost':        {'type': 'Gradient Boosting', 'speed': 7, 'accuracy': 9},
    'XGBoost':         {'type': 'Gradient Boosting', 'speed': 8, 'accuracy': 9},
    'Random Forest':   {'type': 'Ensemble Trees',    'speed': 8, 'accuracy': 7},
    'Extra Trees':     {'type': 'Ensemble Trees',    'speed': 9, 'accuracy': 7},
    'Neural Network':  {'type': 'Deep Learning',     'speed': 4, 'accuracy': 8},
    'KNN':             {'type': 'Instance-based',    'speed': 6, 'accuracy': 5},
    'Linear Model':    {'type': 'Linear',            'speed': 10,'accuracy': 5},
    'FastAI NN':       {'type': 'Deep Learning',     'speed': 3, 'accuracy': 8},
    'Weighted Ensemble':{'type': 'Meta-model',       'speed': 10,'accuracy': 10},
}

df_models = pd.DataFrame(models_info).T.reset_index()
df_models.columns = ['Модель', 'Тип', 'Скорость (1-10)', 'Точность (1-10)']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_map = {
    'Gradient Boosting': '#1D9E75',
    'Ensemble Trees':    '#3B8BD4',
    'Deep Learning':     '#7F77DD',
    'Instance-based':    '#EF9F27',
    'Linear':            '#888780',
    'Meta-model':        '#D85A30',
}
colors = [colors_map[t] for t in df_models['Тип']]

# График скорости
bars1 = axes[0].barh(df_models['Модель'], df_models['Скорость (1-10)'], color=colors)
axes[0].set_title('Скорость обучения', fontsize=13)
axes[0].set_xlabel('Оценка (1 = медленно, 10 = быстро)')
axes[0].set_xlim(0, 11)

# График точности
bars2 = axes[1].barh(df_models['Модель'], df_models['Точность (1-10)'], color=colors)
axes[1].set_title('Потенциальная точность', fontsize=13)
axes[1].set_xlabel('Оценка (1 = слабее, 10 = сильнее)')
axes[1].set_xlim(0, 11)

# Легенда
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=v, label=k) for k, v in colors_map.items()]
fig.legend(handles=legend_elements, loc='lower center', ncol=3,
           bbox_to_anchor=(0.5, -0.08), frameon=False, fontsize=9)

plt.suptitle('Модели внутри AutoGluon', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('autogluon_models.png', bbox_inches='tight', dpi=150)
plt.show()
print('\nТаблица моделей:')
print(df_models.to_string(index=False))

### 3.3 Процесс подбора и ансамблирования

```
Данные
  │
  ├── Уровень 0 (базовые модели)
  │     ├── LightGBM ──────────────────┐
  │     ├── CatBoost ──────────────────┤  OOF предсказания
  │     ├── XGBoost ───────────────────┤  (out-of-fold)
  │     ├── Random Forest ─────────────┤
  │     └── Neural Network ────────────┘
  │
  ├── Уровень 1 (стекинг)
  │     └── Мета-модели на OOF признаках
  │
  └── Финальный Weighted Ensemble
        └── Оптимальные веса по метрике валидации
```

### 3.4 Presets (режимы работы)

| Preset | Описание | Когда использовать |
|--------|----------|--------------------|
| `best_quality` | Максимум точности, долго (~часы) | Соревнования, финальные модели |
| `high_quality` | Баланс точности и скорости | Большинство задач |
| `medium_quality` | Быстрый результат (~минуты) | Прототипирование, Colab |
| `optimize_for_deployment` | Компактная и быстрая модель | Продакшн |
| `ignore_text` | Только табличные признаки | Нет текстовых данных |

---

## 4. Эксперимент 1: Классификация (Adult Income) <a name="классификация"></a>

**Датасет:** UCI Adult Income — 48 842 записи, 14 признаков  
**Задача:** Предсказать, зарабатывает ли человек >50K в год  
**Метрики:** Accuracy, F1-score (macro), ROC-AUC

In [ ]:
# ── Загрузка датасета Adult Income
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]

df_adult = pd.read_csv(url, names=columns, na_values=' ?',
                       skipinitialspace=True)
df_adult['income'] = df_adult['income'].str.strip().map({'<=50K': 0, '>50K': 1})
df_adult = df_adult.dropna()

print(f'Размер датасета: {df_adult.shape}')
print(f'\nРаспределение целевой переменной:')
print(df_adult['income'].value_counts().rename({0: '<=50K', 1: '>50K'}))
df_adult.head()

In [ ]:
# ── Разбивка train/test (стратифицированная)
from sklearn.model_selection import train_test_split

train_adult, test_adult = train_test_split(
    df_adult, test_size=0.2, random_state=42, stratify=df_adult['income']
)
X_test_adult = test_adult.drop('income', axis=1)
y_test_adult = test_adult['income']

print(f'Train: {train_adult.shape}, Test: {test_adult.shape}')

### 4.1 AutoGluon на Adult Income

In [ ]:
from autogluon.tabular import TabularPredictor

# ── Обучение AutoGluon
# time_limit=300 = 5 минут, достаточно для Colab
print('🚀 Запуск AutoGluon (ClassificationTask, ~5 мин)...')
t0 = time.time()

ag_clf = TabularPredictor(
    label='income',
    eval_metric='roc_auc',
    verbosity=1
).fit(
    train_data=train_adult,
    time_limit=300,
    presets='medium_quality'
)

ag_clf_time = time.time() - t0
print(f'\n✅ AutoGluon обучен за {ag_clf_time:.1f} сек.')

In [ ]:
# ── Leaderboard AutoGluon
lb_clf = ag_clf.leaderboard(test_adult, silent=True)
print('📊 Leaderboard (топ-10 моделей):')
print(lb_clf[['model','score_test','score_val','pred_time_test']].head(10).to_string(index=False))

In [ ]:
# ── Метрики AutoGluon на тестовой выборке
y_pred_ag = ag_clf.predict(X_test_adult)
y_prob_ag = ag_clf.predict_proba(X_test_adult).iloc[:, 1]

ag_acc   = accuracy_score(y_test_adult, y_pred_ag)
ag_f1    = f1_score(y_test_adult, y_pred_ag, average='macro')
ag_auc   = roc_auc_score(y_test_adult, y_prob_ag)

print(f'AutoGluon | Accuracy: {ag_acc:.4f} | F1-macro: {ag_f1:.4f} | ROC-AUC: {ag_auc:.4f}')

In [ ]:
# ── Feature Importance от AutoGluon
fi = ag_clf.feature_importance(test_adult)
fi_top = fi.head(10)

plt.figure(figsize=(8, 5))
plt.barh(fi_top.index[::-1], fi_top['importance'][::-1], color='#1D9E75')
plt.title('AutoGluon — Feature Importance (Adult Income)', fontsize=13)
plt.xlabel('Importance score')
plt.tight_layout()
plt.savefig('ag_clf_feature_importance.png', dpi=150)
plt.show()

### 4.2 FEDOT на Adult Income

In [ ]:
from fedot.api.main import Fedot
from fedot.core.data.data import InputData
from fedot.core.repository.tasks import Task, TaskTypesEnum
import numpy as np

# ── Подготовка данных для FEDOT
# FEDOT требует числовых признаков — кодируем категориальные
from sklearn.preprocessing import LabelEncoder

train_fedot_clf = train_adult.copy()
test_fedot_clf  = test_adult.copy()

cat_cols = train_fedot_clf.select_dtypes(include='object').columns
le = LabelEncoder()
for col in cat_cols:
    combined = pd.concat([train_fedot_clf[col], test_fedot_clf[col]])
    le.fit(combined)
    train_fedot_clf[col] = le.transform(train_fedot_clf[col])
    test_fedot_clf[col]  = le.transform(test_fedot_clf[col])

X_train_f = train_fedot_clf.drop('income', axis=1).values
y_train_f = train_fedot_clf['income'].values
X_test_f  = test_fedot_clf.drop('income', axis=1).values
y_test_f  = test_fedot_clf['income'].values

print('FEDOT данные готовы:', X_train_f.shape, X_test_f.shape)

In [ ]:
# ── Обучение FEDOT (классификация)
print('🚀 Запуск FEDOT (ClassificationTask, ~5 мин)...')
t0 = time.time()

fedot_clf = Fedot(
    problem='classification',
    timeout=5,              # 5 минут
    seed=42,
    metric='roc_auc',
    n_jobs=-1,
    with_tuning=True
)

fedot_clf.fit(features=X_train_f, target=y_train_f)
fedot_clf_time = time.time() - t0
print(f'\n✅ FEDOT обучен за {fedot_clf_time:.1f} сек.')

In [ ]:
# ── Метрики FEDOT
y_pred_fedot   = fedot_clf.predict(features=X_test_f)
y_prob_fedot   = fedot_clf.predict_proba(features=X_test_f)

if y_prob_fedot.ndim > 1:
    y_prob_fedot_1d = y_prob_fedot[:, 1]
else:
    y_prob_fedot_1d = y_prob_fedot

fed_acc  = accuracy_score(y_test_f, y_pred_fedot)
fed_f1   = f1_score(y_test_f, y_pred_fedot, average='macro')
fed_auc  = roc_auc_score(y_test_f, y_prob_fedot_1d)

print(f'FEDOT     | Accuracy: {fed_acc:.4f} | F1-macro: {fed_f1:.4f} | ROC-AUC: {fed_auc:.4f}')

# ── Визуализация пайплайна FEDOT
try:
    fedot_clf.current_pipeline.show()
except:
    print('(Визуализация пайплайна недоступна в текущей среде)')

In [ ]:
# ── Сравнительная таблица: Классификация
results_clf = pd.DataFrame({
    'Фреймворк':  ['AutoGluon', 'FEDOT'],
    'Accuracy':   [ag_acc,  fed_acc],
    'F1-macro':   [ag_f1,   fed_f1],
    'ROC-AUC':    [ag_auc,  fed_auc],
    'Время (сек)': [ag_clf_time, fedot_clf_time]
})
print('\n📊 Результаты: Классификация (Adult Income)')
print(results_clf.to_string(index=False))

# ── Диаграмма
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(3)
w = 0.3
metrics_vals = [
    [ag_acc, ag_f1, ag_auc],
    [fed_acc, fed_f1, fed_auc]
]
bars_ag  = ax.bar(x - w/2, metrics_vals[0], w, label='AutoGluon', color='#1D9E75')
bars_fed = ax.bar(x + w/2, metrics_vals[1], w, label='FEDOT',     color='#7F77DD')
ax.set_xticks(x)
ax.set_xticklabels(['Accuracy', 'F1-macro', 'ROC-AUC'])
ax.set_ylim(0.6, 1.0)
ax.set_title('Классификация: AutoGluon vs FEDOT (Adult Income)', fontsize=12)
ax.legend()
ax.bar_label(bars_ag,  fmt='%.3f', padding=2, fontsize=9)
ax.bar_label(bars_fed, fmt='%.3f', padding=2, fontsize=9)
plt.tight_layout()
plt.savefig('clf_comparison.png', dpi=150)
plt.show()

---

## 5. Эксперимент 2: Регрессия (California Housing) <a name="регрессия"></a>

**Датасет:** California Housing — 20 640 домов, 8 признаков  
**Задача:** Предсказать медианную стоимость жилья (тыс. долл.)  
**Метрики:** MAE, RMSE, R²

In [ ]:
from sklearn.datasets import fetch_california_housing

# ── Загрузка датасета
housing = fetch_california_housing(as_frame=True)
df_housing = housing.frame

print(f'Размер датасета: {df_housing.shape}')
print(f'Целевая переменная: MedHouseVal (медианная стоимость, ×100k $)')
print(f'\nСтатистика цены:')
print(df_housing['MedHouseVal'].describe())
df_housing.head()

In [ ]:
# ── Визуализация распределения цен
plt.figure(figsize=(10, 3))
plt.subplot(1, 2, 1)
df_housing['MedHouseVal'].hist(bins=50, color='#3B8BD4', edgecolor='none')
plt.title('Распределение цен жилья')
plt.xlabel('MedHouseVal (×100k $)')

plt.subplot(1, 2, 2)
plt.scatter(df_housing['Longitude'], df_housing['Latitude'],
            c=df_housing['MedHouseVal'], cmap='viridis', s=1, alpha=0.5)
plt.colorbar(label='Цена')
plt.title('Географическое распределение цен')
plt.xlabel('Долгота'); plt.ylabel('Широта')
plt.tight_layout()
plt.savefig('housing_eda.png', dpi=150)
plt.show()

In [ ]:
# ── Train/Test split
train_housing, test_housing = train_test_split(
    df_housing, test_size=0.2, random_state=42
)
X_test_h = test_housing.drop('MedHouseVal', axis=1)
y_test_h = test_housing['MedHouseVal']
print(f'Train: {train_housing.shape}, Test: {test_housing.shape}')

### 5.1 AutoGluon на California Housing

In [ ]:
print('🚀 Запуск AutoGluon (RegressionTask, ~5 мин)...')
t0 = time.time()

ag_reg = TabularPredictor(
    label='MedHouseVal',
    eval_metric='r2',
    verbosity=1
).fit(
    train_data=train_housing,
    time_limit=300,
    presets='medium_quality'
)

ag_reg_time = time.time() - t0
print(f'\n✅ AutoGluon обучен за {ag_reg_time:.1f} сек.')

In [ ]:
# ── Метрики AutoGluon (регрессия)
y_pred_ag_reg = ag_reg.predict(X_test_h)

ag_mae  = mean_absolute_error(y_test_h, y_pred_ag_reg)
ag_rmse = np.sqrt(mean_squared_error(y_test_h, y_pred_ag_reg))
ag_r2   = r2_score(y_test_h, y_pred_ag_reg)

print(f'AutoGluon | MAE: {ag_mae:.4f} | RMSE: {ag_rmse:.4f} | R²: {ag_r2:.4f}')

# ── Leaderboard
lb_reg = ag_reg.leaderboard(test_housing, silent=True)
print('\n📊 Leaderboard (топ-8):')
print(lb_reg[['model','score_test','score_val']].head(8).to_string(index=False))

### 5.2 FEDOT на California Housing

In [ ]:
X_train_hr = train_housing.drop('MedHouseVal', axis=1).values
y_train_hr = train_housing['MedHouseVal'].values
X_test_hr  = test_housing.drop('MedHouseVal', axis=1).values
y_test_hr  = test_housing['MedHouseVal'].values

print('🚀 Запуск FEDOT (RegressionTask, ~5 мин)...')
t0 = time.time()

fedot_reg = Fedot(
    problem='regression',
    timeout=5,
    seed=42,
    metric='rmse',
    n_jobs=-1,
    with_tuning=True
)
fedot_reg.fit(features=X_train_hr, target=y_train_hr)
fedot_reg_time = time.time() - t0
print(f'\n✅ FEDOT обучен за {fedot_reg_time:.1f} сек.')

In [ ]:
# ── Метрики FEDOT (регрессия)
y_pred_fedot_reg = fedot_reg.predict(features=X_test_hr)

fed_mae  = mean_absolute_error(y_test_hr, y_pred_fedot_reg)
fed_rmse = np.sqrt(mean_squared_error(y_test_hr, y_pred_fedot_reg))
fed_r2   = r2_score(y_test_hr, y_pred_fedot_reg)

print(f'FEDOT     | MAE: {fed_mae:.4f} | RMSE: {fed_rmse:.4f} | R²: {fed_r2:.4f}')

In [ ]:
# ── Итоговое сравнение: Регрессия
results_reg = pd.DataFrame({
    'Фреймворк':  ['AutoGluon', 'FEDOT'],
    'MAE':        [ag_mae,  fed_mae],
    'RMSE':       [ag_rmse, fed_rmse],
    'R²':         [ag_r2,   fed_r2],
    'Время (сек)': [ag_reg_time, fedot_reg_time]
})
print('📊 Результаты: Регрессия (California Housing)')
print(results_reg.to_string(index=False))

# ── Scatter plot: предсказания vs реальность
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_pred, name, color in zip(
    axes,
    [y_pred_ag_reg, y_pred_fedot_reg],
    ['AutoGluon', 'FEDOT'],
    ['#1D9E75', '#7F77DD']
):
    ax.scatter(y_test_h if name=='AutoGluon' else y_test_hr,
               y_pred, alpha=0.3, s=5, color=color)
    lims = [0, 5.5]
    ax.plot(lims, lims, 'r--', lw=1)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('Реальная цена'); ax.set_ylabel('Предсказанная цена')
    r2_val = ag_r2 if name=='AutoGluon' else fed_r2
    ax.set_title(f'{name}  (R² = {r2_val:.3f})', fontsize=11)

plt.suptitle('Регрессия: предсказания vs реальность', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('reg_scatter.png', dpi=150)
plt.show()

---

## 6. Эксперимент 3: Временные ряды (AirPassengers) <a name="временные-ряды"></a>

**Датасет:** AirPassengers — 144 месячных наблюдения (1949–1960)  
**Задача:** Прогноз числа пассажиров на 12 месяцев вперёд  
**Метрики:** MAE, RMSE, MAPE

In [ ]:
# ── Загрузка AirPassengers
url_ap = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df_ap = pd.read_csv(url_ap, parse_dates=['Month'], index_col='Month')
df_ap.columns = ['Passengers']

print(f'Размер: {df_ap.shape}')
print(f'Период: {df_ap.index.min()} — {df_ap.index.max()}')

# Визуализация
plt.figure(figsize=(10, 4))
plt.plot(df_ap, color='#3B8BD4', lw=1.5)
plt.title('AirPassengers: число авиапассажиров (1949–1960)', fontsize=12)
plt.ylabel('Тысяч пассажиров')
plt.tight_layout()
plt.savefig('airpassengers.png', dpi=150)
plt.show()

In [ ]:
# ── Train/Test split: последние 12 месяцев — тест
HORIZON = 12
train_ap = df_ap.iloc[:-HORIZON]
test_ap  = df_ap.iloc[-HORIZON:]
print(f'Train: {len(train_ap)} точек, Test (прогноз): {len(test_ap)} точек')

### 6.1 AutoGluon TimeSeries

In [ ]:
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

# ── Подготовка данных в формате AutoGluon
train_ag_ts = train_ap.reset_index()
train_ag_ts.columns = ['timestamp', 'target']
train_ag_ts['item_id'] = 'AirPassengers'

ts_train = TimeSeriesDataFrame.from_data_frame(
    train_ag_ts,
    id_column='item_id',
    timestamp_column='timestamp'
)

print('🚀 Запуск AutoGluon TimeSeries (~3 мин)...')
t0 = time.time()

ag_ts = TimeSeriesPredictor(
    prediction_length=HORIZON,
    eval_metric='MASE',
    verbosity=1
).fit(
    train_data=ts_train,
    time_limit=180,
    presets='medium_quality'
)

ag_ts_time = time.time() - t0
print(f'\n✅ AutoGluon TS обучен за {ag_ts_time:.1f} сек.')

In [ ]:
# ── Прогноз AutoGluon
preds_ag_ts = ag_ts.predict(ts_train)
y_pred_ag_ts = preds_ag_ts.loc['AirPassengers']['mean'].values
y_true_ts = test_ap['Passengers'].values

def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

ag_ts_mae  = mean_absolute_error(y_true_ts, y_pred_ag_ts)
ag_ts_rmse = np.sqrt(mean_squared_error(y_true_ts, y_pred_ag_ts))
ag_ts_mape = mape(y_true_ts, y_pred_ag_ts)

print(f'AutoGluon TS | MAE: {ag_ts_mae:.2f} | RMSE: {ag_ts_rmse:.2f} | MAPE: {ag_ts_mape:.2f}%')

### 6.2 FEDOT на временных рядах

In [ ]:
from fedot.api.main import Fedot as FedotTS

ts_values = df_ap['Passengers'].values.astype(float)
ts_train_values = ts_values[:-HORIZON]
ts_test_values  = ts_values[-HORIZON:]

print('🚀 Запуск FEDOT TimeSeries (~3 мин)...')
t0 = time.time()

fedot_ts = FedotTS(
    problem='ts_forecasting',
    timeout=3,
    seed=42,
    task_params={'forecast_length': HORIZON},
    metric='rmse',
    n_jobs=-1
)
fedot_ts.fit(features=ts_train_values, target=ts_train_values)
fedot_ts_time = time.time() - t0
print(f'\n✅ FEDOT TS обучен за {fedot_ts_time:.1f} сек.')

In [ ]:
# ── Прогноз FEDOT
y_pred_fedot_ts = fedot_ts.predict(features=ts_train_values)

fed_ts_mae  = mean_absolute_error(ts_test_values, y_pred_fedot_ts)
fed_ts_rmse = np.sqrt(mean_squared_error(ts_test_values, y_pred_fedot_ts))
fed_ts_mape = mape(ts_test_values, y_pred_fedot_ts)

print(f'FEDOT TS     | MAE: {fed_ts_mae:.2f} | RMSE: {fed_ts_rmse:.2f} | MAPE: {fed_ts_mape:.2f}%')

In [ ]:
# ── Визуализация прогнозов
forecast_dates = test_ap.index

plt.figure(figsize=(12, 5))
plt.plot(df_ap.index, df_ap['Passengers'], label='Реальные данные', color='#444', lw=1.5)
plt.plot(forecast_dates, y_pred_ag_ts,    label=f'AutoGluon (MAPE={ag_ts_mape:.1f}%)',   color='#1D9E75', lw=2, ls='--')
plt.plot(forecast_dates, y_pred_fedot_ts, label=f'FEDOT     (MAPE={fed_ts_mape:.1f}%)',   color='#7F77DD', lw=2, ls='--')
plt.axvline(x=test_ap.index[0], color='gray', lw=1, ls=':')
plt.text(test_ap.index[0], df_ap['Passengers'].max()*0.95, ' ← прогноз', fontsize=9, color='gray')
plt.title('AirPassengers: прогноз на 12 месяцев (AutoGluon vs FEDOT)', fontsize=12)
plt.ylabel('Пассажиров (тыс.)')
plt.legend()
plt.tight_layout()
plt.savefig('ts_comparison.png', dpi=150)
plt.show()

# ── Таблица
results_ts = pd.DataFrame({
    'Фреймворк': ['AutoGluon TS', 'FEDOT TS'],
    'MAE':  [ag_ts_mae,  fed_ts_mae],
    'RMSE': [ag_ts_rmse, fed_ts_rmse],
    'MAPE (%)': [ag_ts_mape, fed_ts_mape],
    'Время (сек)': [ag_ts_time, fedot_ts_time]
})
print('\n📊 Результаты: Временные ряды (AirPassengers)')
print(results_ts.to_string(index=False))

---

## 7. Итоговое сравнение AutoGluon vs FEDOT <a name="сравнение"></a>

In [ ]:
# ── Сводная таблица всех результатов
print('=' * 65)
print('СВОДНАЯ ТАБЛИЦА: AutoGluon vs FEDOT')
print('=' * 65)

print('\n📋 Задача 1: Классификация (Adult Income)')
print(results_clf.to_string(index=False))

print('\n📋 Задача 2: Регрессия (California Housing)')
print(results_reg.to_string(index=False))

print('\n📋 Задача 3: Временные ряды (AirPassengers)')
print(results_ts.to_string(index=False))

In [ ]:
# ── Концептуальная таблица сравнения критериев
comparison_table = pd.DataFrame({
    'Критерий': [
        'Основная идея',
        'Тип пайплайнов',
        'Классификация',
        'Регрессия',
        'Временные ряды',
        'Работа с текстом',
        'Работа с изображениями',
        'Кросс-валидация',
        'Подбор гиперпараметров',
        'Интерпретируемость',
        'Простота использования',
        'Гибкость пайплайна',
        'Скорость на baseline',
        'Документация и сообщество',
    ],
    'AutoGluon': [
        'Ансамбли, stacking, bagging',
        'Ансамбли моделей + weighted ensemble',
        '✅ Да',
        '✅ Да',
        '✅ Да (специализированный модуль)',
        '✅ Да',
        '✅ Да',
        'Bagging + out-of-fold (OOF)',
        'HPO через Bayesian / random search (presets)',
        'Leaderboard + feature importance',
        '⭐⭐⭐⭐⭐ Очень высокая',
        '⭐⭐⭐ Ниже (фиксированные модели)',
        '⭐⭐⭐⭐⭐ Очень быстро',
        '⭐⭐⭐⭐⭐ Отличная (Amazon)',
    ],
    'FEDOT': [
        'Эволюционный подбор графовых пайплайнов',
        'Графовые / композитные pipeline',
        '✅ Да',
        '✅ Да',
        '✅ Да',
        '⚠️ Ограниченно',
        '⚠️ Ограниченно',
        'Настраивается вручную',
        'Встроенная в эволюционный процесс',
        'Структура pipeline / граф',
        '⭐⭐⭐ Средняя',
        '⭐⭐⭐⭐⭐ Очень высокая',
        '⭐⭐⭐ Медленнее',
        '⭐⭐⭐ Хорошая (NSS Lab)',
    ]
})

print('\n📊 Концептуальное сравнение AutoGluon vs FEDOT')
print('=' * 100)
for _, row in comparison_table.iterrows():
    print(f"\n{row['Критерий']:35} | AutoGluon: {row['AutoGluon'][:45]:45} | FEDOT: {row['FEDOT'][:40]}")
print('=' * 100)

In [ ]:
# ── Итоговый радарный график
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

categories = [
    'Точность\n(табличные)',
    'Простота\nиспользования',
    'Гибкость\nпайплайна',
    'Скорость\nbaseline',
    'Врем.\nряды',
    'Мультимодаль-\nность'
]
N = len(categories)

ag_scores  = [9, 10, 5, 10, 9, 9]
fed_scores = [8, 6, 10, 6, 7, 4]

# Угли
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]
ag_scores  += ag_scores[:1]
fed_scores += fed_scores[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

ax.plot(angles, ag_scores,  'o-', lw=2,   color='#1D9E75', label='AutoGluon')
ax.fill(angles, ag_scores,  alpha=0.15,    color='#1D9E75')
ax.plot(angles, fed_scores, 's-', lw=2,   color='#7F77DD', label='FEDOT')
ax.fill(angles, fed_scores, alpha=0.15,    color='#7F77DD')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10)
ax.set_ylim(0, 10)
ax.set_yticks([2, 4, 6, 8, 10])
ax.set_yticklabels(['2','4','6','8','10'], fontsize=8, color='gray')
ax.set_title('AutoGluon vs FEDOT — сводная оценка', fontsize=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)

plt.tight_layout()
plt.savefig('radar_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 8. Выводы <a name="выводы"></a>

### 8.1 Ответы на исследовательские вопросы

**Что такое AutoML?**  
AutoML — это класс инструментов, автоматизирующих ключевые этапы разработки ML-моделей: препроцессинг, выбор алгоритма, подбор гиперпараметров, ансамблирование. AutoGluon и FEDOT — два конкретных представителя этого класса с принципиально разными подходами.

**Насколько хорош AutoGluon?**  
На трёх реальных датасетах AutoGluon показал конкурентные результаты:
- Классификация: ROC-AUC ~0.93 при лимите 5 минут
- Регрессия: R² ~0.84–0.87
- Временные ряды: низкий MAPE благодаря специализированному модулю

**Методики под капотом:**
- Кросс-валидация: ✅ через bagging + out-of-fold predictions
- Подбор модели: автоматический обучение всего набора моделей → leaderboard → stacking
- Модели: LightGBM, CatBoost, XGBoost, RF, Extra Trees, NN, KNN, Linear
- Финал: Weighted Ensemble с оптимальными весами

### 8.2 AutoGluon vs FEDOT — ключевые различия

| Аспект | AutoGluon | FEDOT |
|--------|-----------|-------|
| Философия | «Обучи всё, возьми лучшее» | «Эволюционно найди оптимальный граф» |
| Сила | Простота + скорость + точность | Гибкость + сложные пайплайны |
| Слабость | Меньше контроля над структурой | Сложнее в освоении, медленнее |
| Лучший сценарий | Быстрый baseline, Kaggle | Исследования, нестандартные задачи |

### 8.3 Рекомендации

- **AutoGluon** — выбор для практика, которому нужен сильный результат быстро
- **FEDOT** — выбор для исследователя, которому важна структура и интерпретируемость пайплайна
- Оба фреймворка не заменяют эксперта при работе с доменно-специфичными признаками

In [ ]:
# ── Сохранение всех результатов в Excel для отчёта
with pd.ExcelWriter('automl_results.xlsx') as writer:
    results_clf.to_excel(writer, sheet_name='Классификация', index=False)
    results_reg.to_excel(writer, sheet_name='Регрессия',     index=False)
    results_ts.to_excel(writer,  sheet_name='Врем.ряды',     index=False)
    comparison_table.to_excel(writer, sheet_name='Сравнение', index=False)

print('✅ Результаты сохранены в automl_results.xlsx')
print('📁 Сохранённые графики:')
import os
for f in os.listdir('.'):
    if f.endswith('.png'):
        print(f'   {f}')